In [ ]:
#Import Path and import libraries
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

pd.set_option("display.max_columns",None)
pd.set_option("display.width",140)

#Define the path to the raw dataset
DATA_PATH=Path("../data/raw/car_price_dataset.csv")
TABLE_DIR=Path("../reports/tables")

TABLE_DIR.mkdir(parents=True,exist_ok=True)

#Load the CSV dataset into a pandas DataFrame
df=pd.read_csv(DATA_PATH)

#Number of rows and columns in the dataset
print(f"Dataset shape: {df.shape[0]:,}rows*{df.shape[1]}columns")

#Top 5 rows of the dataset
display(df.head())

Dataset shape: 9,788rows*17columns


,Unnamed: 0,Brand,Model,YOM,Engine (cc),Gear,Fuel Type,Millage(KM),Town,Date,Leasing,Condition,AIR CONDITION,POWER STEERING,POWER MIRROR,POWER WINDOW,Price
0,0,AUDI,A1,2016.0,990.0,Automatic,Petrol,99000.0,Gampaha,2/5/2025,No Leasing,USED,Available,Available,Available,Available,100.0
1,1,AUDI,A1,2017.0,1000.0,Automatic,Petrol,88000.0,Colombo,1/14/2025,No Leasing,USED,Available,Available,Available,Available,97.0
2,2,AUDI,A1,2018.0,1000.0,Automatic,Petrol,77000.0,Dehiwala-Mount-Lavinia,1/23/2025,No Leasing,USED,Available,Available,Available,Available,98.5
3,3,AUDI,A1,2017.0,1000.0,Automatic,Petrol,88000.0,Negombo,12/21/2024,No Leasing,USED,Available,Available,Available,Available,107.0
4,4,AUDI,A1,2017.0,1000.0,Automatic,Petrol,88000.0,Colombo,12/21/2024,No Leasing,USED,Available,Available,Available,Available,99.5


In [ ]:
print("Columns:")

#Loop through al column names in the DataFrame
for i,column in enumerate(df.columns,1):
    print(f"{i:02d}.{column}")

print("\nData Types:")
display(df.dtypes.rename("dtype").to_frame())

print("\nDataset information")
#Display detailed information fabout the DataFrame
df.info()

Columns:
01.Unnamed: 0
02.Brand
03.Model
04.YOM
05.Engine (cc)
06.Gear
07.Fuel Type
08.Millage(KM)
09.Town
10.Date
11.Leasing
12.Condition
13.AIR CONDITION
14.POWER STEERING
15.POWER MIRROR
16.POWER WINDOW
17.Price

Data Types:


,dtype
Unnamed: 0,int64
Brand,str
Model,str
YOM,float64
Engine (cc),float64
Gear,str
Fuel Type,str
Millage(KM),float64
Town,str
Date,str



Dataset information
<class 'pandas.DataFrame'>
RangeIndex: 9788 entries, 0 to 9787
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      9788 non-null   int64  
 1   Brand           9788 non-null   str    
 2   Model           9788 non-null   str    
 3   YOM             9776 non-null   float64
 4   Engine (cc)     9779 non-null   float64
 5   Gear            9788 non-null   str    
 6   Fuel Type       9788 non-null   str    
 7   Millage(KM)     9779 non-null   float64
 8   Town            9788 non-null   str    
 9   Date            9788 non-null   str    
 10  Leasing         9788 non-null   str    
 11  Condition       9788 non-null   str    
 12  AIR CONDITION   9788 non-null   str    
 13  POWER STEERING  9788 non-null   str    
 14  POWER MIRROR    9788 non-null   str    
 15  POWER WINDOW    9788 non-null   str    
 16  Price           9765 non-null   float64
dtypes: float64(4), int64(1)

In [5]:
TARGET="Price"

print("Target variable:",TARGET)
print("Target Data",df[TARGET].dtype)
print("Missing target values:",df[TARGET].isna().sum())

display(
    df[TARGET].describe(
        percentiles=[0.01,0.05,0.25,0.50,0.75,0.90,0.95,0.99]
    )
)

Target variable: Price
Target Data float64
Missing target values: 23


count    9765.000000
mean       53.414791
std        43.642370
min         2.650000
1%          6.232000
5%         10.500000
25%        26.250000
50%        43.000000
75%        69.560417
90%        95.000000
95%       125.000000
99%       225.000000
max       790.000000
Name: Price, dtype: float64

In [6]:
profile=pd.DataFrame({
    "dtype":df.dtypes.astype(str),
    "missing_count":df.isna().sum(),
    "missing_pct":(df.isna().mean()*100).round(3),
    "n_unique":df.nunique(dropna=True),
    "sample_values":[
        ",".join(map(str,df[col].dropna().unique()[:5]))
        for col in df.columns
    ]
})

display(
    profile.sort_values(
        ["missing_count","n_unique"],
        ascending=[False,False]
    )
)

profile.to_csv(
    TABLE_DIR/"data_profile.csv"
)

,dtype,missing_count,missing_pct,n_unique,sample_values
Price,float64,23,0.235,1684,"100.0,97.0,98.5,107.0,99.5"
YOM,float64,12,0.123,62,"2016.0,2017.0,2018.0,2015.0,2001.0"
Engine (cc),float64,9,0.092,278,"990.0,1000.0,1500.0,1750.0,999.0"
Millage(KM),float64,9,0.092,62,"99000.0,88000.0,77000.0,110000.0,264000.0"
Unnamed: 0,int64,0,0.000,9784,"0,1,2,3,4"
Model,str,0,0.000,1554,"A1,A1 RS,A3,A3 S LINE,A3 S-LINE"
Town,str,0,0.000,107,"Gampaha,Colombo,Dehiwala-Mount-Lavinia,Negombo..."
Date,str,0,0.000,65,"2/5/2025,1/14/2025,1/23/2025,12/21/2024,12/16/..."
Brand,str,0,0.000,50,"AUDI,AUSTIN,BAJAJ,BMW,BYD"
Fuel Type,str,0,0.000,4,"Petrol,Diesel,Hybrid,Electric"


In [8]:
missing=(
    df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

missing=missing[
    missing["missing_count"]>0
].sort_values(
    "missing_count",
    ascending=False
)

print(
    "Rows with at least one missing values:",
    df.isna().any(axis=1).sum()
)
display(missing)

Rows with at least one missing values: 53


,missing_count
Price,23
YOM,12
Engine (cc),9
Millage(KM),9


In [10]:
df.loc[
    df.isna().any(axis=1)
].to_csv(
    TABLE_DIR / "rows_with_missing_values.csv",
    index=False
)

In [12]:
exact_duplicates=df.duplicated().sum()
duplicate_ids=df["Unnamed: 0"].duplicated().sum()

#Display number of exact duplicate values
print("Exact duplicate rows:",exact_duplicates)
#Display number of duplicated ID values
print("Duplicated values is Unnamed: 0:",duplicate_ids)

Exact duplicate rows: 4
Duplicated values is Unnamed: 0: 4


In [13]:
duplicates = df[
    df.duplicated(keep=False)
].sort_values("Unnamed: 0")

display(duplicates)

,Unnamed: 0,Brand,Model,YOM,Engine (cc),Gear,Fuel Type,Millage(KM),Town,Date,Leasing,Condition,AIR CONDITION,POWER STEERING,POWER MIRROR,POWER WINDOW,Price
69,70,AUSTIN,MINI COOPER,1970.0,1300.0,Manual,Petrol,605000.0,Kandy,12/11/2024,No Leasing,USED,Available,Not_Available,Not_Available,Not_Available,25.25
70,70,AUSTIN,MINI COOPER,1970.0,1300.0,Manual,Petrol,605000.0,Kandy,12/11/2024,No Leasing,USED,Available,Not_Available,Not_Available,Not_Available,25.25
167,167,CHERY,QQ,2007.0,812.0,Manual,Petrol,198000.0,Kalutara,2/4/2025,No Leasing,USED,Available,Available,Available,Available,13.65
170,167,CHERY,QQ,2007.0,812.0,Manual,Petrol,198000.0,Kalutara,2/4/2025,No Leasing,USED,Available,Available,Available,Available,13.65
259,261,DAIHATSU,CHARADE,1991.0,1000.0,Manual,Petrol,374000.0,Matara,1/11/2025,No Leasing,USED,Available,Not_Available,Not_Available,Not_Available,12.00
261,261,DAIHATSU,CHARADE,1991.0,1000.0,Manual,Petrol,374000.0,Matara,1/11/2025,No Leasing,USED,Available,Not_Available,Not_Available,Not_Available,12.00
8987,8989,TOYOTA,STARLET,1997.0,1300.0,Automatic,Petrol,308000.0,Kuliyapitiya,12/16/2024,No Leasing,USED,Available,Available,Available,Available,36.75
8988,8989,TOYOTA,STARLET,1997.0,1300.0,Automatic,Petrol,308000.0,Kuliyapitiya,12/16/2024,No Leasing,USED,Available,Available,Available,Available,36.75


In [14]:
numeric_columns=df.select_dtypes(
    include=np.number
).columns.tolist()

display(
    df[numeric_columns].describe(
         percentiles=[
            .01, .05, .25, .50,
            .75, .90, .95, .99
        ]
    ).T
)

,count,mean,std,min,1%,5%,25%,50%,75%,90%,95%,99%,max
Unnamed: 0,9788.0,4893.759093,2826.029139,0.00,97.870,489.35,2446.75,4893.5,7341.250000,8809.3,9298.65,9690.13,9788.0
YOM,9776.0,2006.757876,9.837492,1956.00,1980.000,1987.00,2001.00,2009.0,2015.000000,2017.0,2018.00,2019.00,2024.0
Engine (cc),9779.0,1260.375478,403.884873,573.00,650.000,650.00,1000.00,1300.0,1500.000000,1700.0,1997.00,2342.20,4800.0
Millage(KM),9779.0,200679.415073,108185.742286,11000.00,66000.000,77000.00,110000.00,176000.0,264000.000000,363000.0,418000.00,495000.00,759000.0
Price,9765.0,53.414791,43.642370,2.65,6.232,10.50,26.25,43.0,69.560417,95.0,125.00,225.00,790.0


In [16]:
for column in [
    "YOM",
    "Engine (cc)",
    "Millage(KM)",
    "Price"
]:

    print(f"\nLowest 5 — {column}")

    display(
        df.nsmallest(
            5, column
        )[[
            "Brand",
            "Model",
            column,
            "Price"
        ]]
    )

    print(f"Highest 5 — {column}")

    display(
        df.nlargest(
            5, column
        )[[
            "Brand",
            "Model",
            column,
            "Price"
        ]]
    )


Lowest 5 — YOM


,Brand,Model,YOM,Price
3887,PEUGEOT,203,1956.0,40.00
2371,MORRIS,MINOR,1959.0,24.00
56,AUSTIN,CAMBRIDGE,1960.0,12.75
58,AUSTIN,MINI,1960.0,25.00
64,AUSTIN,MINI COOPER,1960.0,22.95


Highest 5 — YOM


,Brand,Model,YOM,Price
55,AUDI,Q4,2024.0,475.0
159,BYD,SEAL,2024.0,394.5
242,CITROEN,E-C4,2024.0,228.5
297,DAIHATSU,MIRA,2024.0,59.5
819,HONDA,FIT GP5,2024.0,81.0



Lowest 5 — Engine (cc)


...

,Brand,Model,Millage(KM),Price
3887,PEUGEOT,203,759000.0,40.00
2371,MORRIS,MINOR,726000.0,24.00
56,AUSTIN,CAMBRIDGE,715000.0,12.75
58,AUSTIN,MINI,715000.0,25.00
64,AUSTIN,MINI COOPER,715000.0,22.95



Lowest 5 — Price


,Brand,Model,Price,Price
8401,TOYOTA,KE50,2.65,2.65
2103,MITSUBISHI,COLT,3.25,3.25
2131,MITSUBISHI,GALANT,3.40,3.40
2488,NISSAN,B310,3.65,3.65
2165,MITSUBISHI,LANCER,3.75,3.75


Highest 5 — Price


,Brand,Model,Price,Price
8417,TOYOTA,LEXUS 600HL,790.0,790.0
138,BMW,740LE M SPORT,575.0,575.0
1776,MERCEDES-BENZ,S400 LONG WHEEL BASE,550.0,550.0
55,AUDI,Q4,475.0,475.0
133,BMW,530I M SPORT,460.0,460.0


In [18]:
categorical_columns=df.select_dtypes(
    include="object"
).columns.tolist()

cardinality=pd.DataFrame({
    "n_unique":[
        df[col].nunique()
        for col in categorical_columns
    ],
    "top_category_share_pct":[
        round(
            df[col].value_counts(normalize=True).iloc[0]*100,
            2
        )
        for col in categorical_columns
    ]
})

cardinality.index=categorical_columns

display(
    cardinality.sort_values(
        "n_unique",
        ascending=False
    )
)

D:\Temp\ipykernel_7192\3384449318.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns=df.select_dtypes(


,n_unique,top_category_share_pct
Model,1554,4.00
Town,107,10.60
Date,65,6.58
Brand,50,31.56
Fuel Type,4,80.36
Gear,2,64.41
Leasing,2,95.87
Condition,2,99.04
AIR CONDITION,2,97.99
POWER STEERING,2,91.10


In [19]:
model_counts = df["Model"].value_counts()

print("Unique models:", model_counts.size)

print(
    "Models appearing once:",
    (model_counts == 1).sum()
)

print(
    "Models appearing at least 5 times:",
    (model_counts >= 5).sum()
)

print(
    "Models appearing at least 10 times:",
    (model_counts >= 10).sum()
)

Unique models: 1554
Models appearing once: 914
Models appearing at least 5 times: 296
Models appearing at least 10 times: 177


In [20]:
town_counts = df["Town"].value_counts()

print("Unique towns:", town_counts.size)

print(
    "Singleton towns:",
    (town_counts == 1).sum()
)

display(
    town_counts.head(20)
)

Unique towns: 107
Singleton towns: 3


Town
Colombo                   1038
Gampaha                    655
Kurunegala                 646
Kandy                      638
Matara                     320
Malabe                     273
Galle                      253
Kadawatha                  235
Negombo                    228
Anuradapura                212
Homagama                   212
Dehiwala-Mount-Lavinia     189
Kegalle                    186
Panadura                   183
Kalutara                   169
Kuliyapitiya               161
Nugegoda                   160
Piliyandala                156
Horana                     149
Ratnapura                  147
Name: count, dtype: int64

In [25]:
df["Data_parsed"]=pd.to_datetime(
    df["Date"],
    errors="coerce"
)

print(
    "Date range:",
    df["Data_parsed"].min(),
    "to",
    df["Data_parsed"].max()
)
print(
    "Distinct dates:",
    df["Data_parsed"].nunique()
)
print(
    "Unparseable dates:",
    df["Data_parsed"].isna().sum()
)

Date range: 2024-12-03 00:00:00 to 2025-02-05 00:00:00
Distinct dates: 65
Unparseable dates: 0


In [26]:
pair = df[
    ["YOM", "Millage(KM)"]
].dropna()

print(
    "Maximum unique mileage values for one YOM:",
    pair.groupby("YOM")["Millage(KM)"].nunique().max()
)

Maximum unique mileage values for one YOM: 1


In [28]:
model = LinearRegression()

model.fit(
    pair[["YOM"]],
    pair["Millage(KM)"]
)

print(
    f"Mileage = "
    f"{model.coef_[0]:,.0f} × YOM + "
    f"{model.intercept_:,.0f}"
)

print(
    "R² =",
    model.score(
        pair[["YOM"]],
        pair["Millage(KM)"]
    )
)

Mileage = -11,000 × YOM + 22,275,000
R² = 1.0
